# Recovered Colab notebook — sanitized historical archive

Recovered on 2026-10-02. This is historical code, including superseded model experiments; it is not the maintained reproduction entry point. Credentials and passage-bearing outputs were removed. Only cleaning counts, split counts, and the MLE audit stream output are retained. Null execution counters do not establish original execution order. Use the numbered notebooks for current work.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a project directory if it doesn't exist
PROJECT_PATH = '/content/drive/MyDrive/darija_project'
os.makedirs(PROJECT_PATH, exist_ok=True)

print(f'Drive mounted. Project directory ready at: {PROJECT_PATH}')

This cell clears any existing model or dataframe variables from memory to ensure a clean slate for execution, particularly useful when running the notebook multiple times.

# Moroccan Arabizi-to-Arabic Transliteration Pipeline
This notebook contains the complete pipeline for benchmarking Rule-Based, MLE, and Neural (ByT5) models.

**Warning:** Running this will reset the environment and start a fresh training run.

In [ ]:
import gc
import torch

# Reset the environment for a clean 'Run All'
try:
    del model, trainer, parallel_df, train_df, val_df, test_df, tokenized_datasets
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Environment reset. Ready for clean small-scale execution.')

### Phase 1: Environment & Authentication
First, we install the core libraries and handle authentication to access the gated NileChat datasets.

This cell installs all necessary Python libraries with specific version pinning to resolve compatibility issues and authenticates with Hugging Face Hub using a provided token to access private datasets. It also includes aggressive uninstall commands to ensure a clean environment.

In [ ]:
import os
import gc
import torch
import pandas as pd
import re
from huggingface_hub import login

# Pinning versions for compatibility
!pip install --upgrade pip --quiet # Ensure pip is up-to-date

# Uninstall critical conflicting packages to ensure a clean slate
# Including all packages mentioned in conflict reports, plus common HF-ecosystem libraries (e.g., diffusers, gradio)
!pip uninstall -y transformers datasets sacrebleu accelerate huggingface_hub numpy tokenizers camel-tools torchtune peft sentence-transformers shap cuml-cu12 pytensor quantecon umap-learn segregation librosa cudf-cu12 pynndescent stumpy diffusers gradio --quiet

# Install huggingface_hub to a known compatible older version for transformers==4.26.1
!pip install huggingface_hub==0.11.0 --quiet

# Install a very stable, older accelerate version to avoid fp8_capability conflicts
!pip install accelerate==0.15.0 --quiet --force-reinstall

# Install transformers to a version known to work with accelerate==0.15.0
!pip install transformers==4.26.1 datasets sacrebleu --quiet

# Verify installed versions
print("---" * 10)
print("--- Installed package versions ---")
!pip show accelerate
!pip show transformers
!pip show huggingface_hub

# Explicitly import and print accelerate version and its file path to confirm what Python loads
import accelerate
print(f"Accelerate module loaded version: {accelerate.__version__}")
print(f"Accelerate module loaded from: {accelerate.__file__}")

print("---" * 10)

# Clear memory
gc.collect()
torch.cuda.empty_cache()

# Authentication
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)
print("Core libraries installed and environment set up.")

This cell streams Arabizi and Arabic datasets in batches to manage memory, merges them based on a common ID, and then concatenates the processed chunks into a single DataFrame. It also defines a `SEARCH_BUFFER` for later use.

### Phase 2: Data Retrieval & Alignment
In this section, we fetch parallel data from the NileChat datasets using streaming mode. We align the Arabizi and Arabic scripts using their unique message IDs to create a parallel corpus.

In [ ]:
import pandas as pd
import gc
import os

# Ensure PROJECT_PATH is defined (from cell 3ccd41ed)
PROJECT_PATH = '/content/drive/MyDrive/darija_project'
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv') # Define path for saved data

# --- Load from saved CSV only (no Hugging Face streaming fallback) ---
if os.path.exists(CLEAN_DATA_PATH):
    parallel_df = pd.read_csv(CLEAN_DATA_PATH)
    print(f"Data loaded from save-point: {len(parallel_df)} aligned rows from {CLEAN_DATA_PATH}.")
    # In this case, SEARCH_BUFFER represents the length of the loaded, already cleaned data
    SEARCH_BUFFER = len(parallel_df)
else:
    raise FileNotFoundError(f"No save-point found at {CLEAN_DATA_PATH}. Please ensure it has been created by running previous phases.")

display(parallel_df.head())

This cell verifies the total number of aligned rows in the `parallel_df` and displays the first 10 entries to ensure data was loaded and merged correctly.

In [ ]:
# Verify the new scale of the aligned dataset
print(f'Total aligned rows is: {len(parallel_df)}')
display(parallel_df.head(10))

This cell applies a series of cleaning operations to the parallel dataframe: removing URLs, normalizing character repetitions, filtering out Arabizi entries that contain Arabic script, and applying a length-ratio filter based on word counts to improve data quality.

### Phase 3: Preprocessing & Cleaning
We implement regex cleaning to remove noise, a length-ratio filter to ensure alignment quality, and character normalization.

In [ ]:
import re
import gc # Import gc for explicit garbage collection

def clean_text(text):
    if not isinstance(text, str): return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Character repetition normalization (e.g., haaaaa -> haa)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Basic punctuation cleaning
    text = re.sub(r'[!?,\-\n]', ' ', text)
    return text.strip()

def contains_arabic(text):
    return bool(re.search('[\u0600-\u06FF]', str(text)))

# Safety check: only proceed if we have data
if not parallel_df.empty and 'text_az' in parallel_df.columns:
    print(f"Initial rows before cleaning: {len(parallel_df)}")

    # Apply cleaning
    parallel_df['text_az'] = parallel_df['text_az'].apply(clean_text)
    parallel_df['text_ar'] = parallel_df['text_ar'].apply(clean_text)
    gc.collect() # Free memory after creating new series

    # Discard rows where Arabizi column actually contains Arabic script
    initial_rows_after_cleaning = len(parallel_df)
    parallel_df = parallel_df[~parallel_df['text_az'].apply(contains_arabic)]
    print(f"Rows after filtering Arabic in Arabizi: {len(parallel_df)} (Removed {initial_rows_after_cleaning - len(parallel_df)})")
    gc.collect() # Free memory after filtering

    # Word counts for length-ratio filtering
    # Create these columns
    parallel_df['len_az'] = parallel_df['text_az'].str.split().str.len()
    parallel_df['len_ar'] = parallel_df['text_ar'].str.split().str.len()
    gc.collect() # Free memory after creating new series

    # Avoid division by zero and perform initial length filter
    initial_rows_before_zero_filter = len(parallel_df)
    parallel_df = parallel_df[(parallel_df['len_az'] > 0) & (parallel_df['len_ar'] > 0)]
    print(f"Rows after removing zero-length sentences: {len(parallel_df)} (Removed {initial_rows_before_zero_filter - len(parallel_df)})")
    gc.collect() # Free memory after filtering

    # Length-Ratio Filter: Discard if word count difference is > 2x
    parallel_df['ratio'] = parallel_df['len_az'] / parallel_df['len_ar']
    gc.collect() # Free memory after creating new series

    initial_rows_before_ratio_filter = len(parallel_df)
    parallel_df = parallel_df[(parallel_df['ratio'] >= 0.5) & (parallel_df['ratio'] <= 2.0)]
    print(f"Rows after length-ratio filter: {len(parallel_df)} (Removed {initial_rows_before_ratio_filter - len(parallel_df)})")
    gc.collect() # Free memory after filtering

    # Explicitly drop temporary columns
    parallel_df = parallel_df.drop(columns=['len_az', 'len_ar', 'ratio'], errors='ignore')
    gc.collect() # Free memory after dropping columns

    print(f"Dataset cleaned. Remaining rows: {len(parallel_df)}")
    display(parallel_df.head())
else:
    print("Warning: parallel_df is empty or missing columns. Cannot perform cleaning.")

Initial rows before cleaning: 8045
Rows after filtering Arabic in Arabizi: 8045 (Removed 0)
Rows after removing zero-length sentences: 8045 (Removed 0)
Rows after length-ratio filter: 8045 (Removed 0)
Dataset cleaned. Remaining rows: 8045


This cell performs a critical final cleaning step by dropping any rows that contain null values or empty strings in either the Arabizi or Arabic text columns, preventing potential issues in subsequent processing stages.

In [ ]:
# Critical Fix: Remove nulls and empty strings to prevent 'nan' loss
initial_len = len(parallel_df)
parallel_df = parallel_df.dropna(subset=['text_az', 'text_ar'])
parallel_df = parallel_df[(parallel_df['text_az'].astype(str).str.strip() != "") & (parallel_df['text_ar'].astype(str).str.strip() != "")]

print(f"Rows before final cleaning: {initial_len}")
print(f"Rows after final cleaning: {len(parallel_df)}")
print(f"Rows removed: {initial_len - len(parallel_df)}")

This cell persists the cleaned parallel corpus to a CSV file in Google Drive, creating a save-point that allows for recovery in case of runtime crashes or to resume work without re-running previous data loading and cleaning phases.

In [ ]:
# Step 2: Create 'Save-Point' for the parallel corpus
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv')

# Save to Drive
parallel_df.to_csv(CLEAN_DATA_PATH, index=False)
print(f'Persistence Success: Cleaned data saved to {CLEAN_DATA_PATH}')

This cell visualizes the data pipeline's volume and health, illustrating the number of rows at different stages: the raw stream, after ID alignment, and after cleaning and filtering. This helps track data loss during preprocessing.

### Phase 4: Dataset Partitioning
We split the cleaned corpus into training, validation, and testing sets (80/10/10). This ensures that our evaluation metrics, like BLEU, are calculated on unseen data.

### Recovery: Load from Save-Point
Run this cell if your runtime crashed and you want to skip Phase 2 & 3.

In [ ]:
import pandas as pd
import os
PROJECT_PATH = '/content/drive/MyDrive/darija_project'
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv')

if os.path.exists(CLEAN_DATA_PATH):
    parallel_df = pd.read_csv(CLEAN_DATA_PATH)
    print(f'Data recovered: {len(parallel_df)} rows loaded.')
else:
    print('No save-point found. Please run Phase 2 and 3 first.')

This cell splits the cleaned dataset (`parallel_df`) into training, validation, and test sets using an 80/10/10 ratio. It then saves these splits as CSV files to Google Drive for consistent access by subsequent algorithms.

#### Visualization: Data Health (Phase 2 & 3)
We track the filtering process and analyze the distribution of our final corpus.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# Retrieve counts from saved files if parallel_df is missing
try:
    if 'parallel_df' in globals() and not parallel_df.empty:
        current_count = len(parallel_df)
    else:
        # Fallback to sum of split files if parallel_df is not in memory
        PROJECT_PATH = '/content/drive/MyDrive/darija_project'
        train_c = len(pd.read_csv(os.path.join(PROJECT_PATH, 'toy_train.csv'))) if os.path.exists(os.path.join(PROJECT_PATH, 'toy_train.csv')) else 0
        val_c = len(pd.read_csv(os.path.join(PROJECT_PATH, 'toy_val.csv'))) if os.path.exists(os.path.join(PROJECT_PATH, 'toy_val.csv')) else 0
        test_c = len(pd.read_csv(os.path.join(PROJECT_PATH, 'toy_test.csv'))) if os.path.exists(os.path.join(PROJECT_PATH, 'toy_test.csv')) else 0
        current_count = train_c + val_c + test_c
except Exception as e:
    print(f"Error determining current_count: {e}")
    current_count = 0 # Default to 0 if an error occurs

# Data Cleaning Drop-off Stats
stats = {
    'Raw Stream': SEARCH_BUFFER, # SEARCH_BUFFER from the data loading cell
    'Merged (ID Alignment)': SEARCH_BUFFER, # Updated to dynamically use SEARCH_BUFFER
    'Cleaned & Filtered': current_count
}

plt.figure(figsize=(10, 6))
sns.barplot(x=list(stats.keys()), y=list(stats.values()), palette='viridis', hue=list(stats.keys()), legend=False)

# Add numerical labels on top of bars
for i, v in enumerate(stats.values()):
    plt.text(i, v + (max(stats.values()) * 0.01), str(int(v)), ha='center', fontweight='bold')

plt.title('Figure 1: Data Pipeline Volume & Health')
plt.ylabel('Number of Rows')
plt.show()

### Phase 4: The Scientific Split
We split our cleaned toy dataset into Train, Validation, and Test sets to ensure a robust evaluation framework.

In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd

# Ensure PROJECT_PATH is defined
if 'PROJECT_PATH' not in globals():
    PROJECT_PATH = '/content/drive/MyDrive/darija_project'

# Ensure parallel_df is loaded if not already in memory
if 'parallel_df' not in globals():
    print("parallel_df not found in RAM. Attempting to load from save-point...")
    CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv')
    if os.path.exists(CLEAN_DATA_PATH):
        parallel_df = pd.read_csv(CLEAN_DATA_PATH)
        print(f'Data loaded: {len(parallel_df)} rows from {CLEAN_DATA_PATH}.')
    else:
        raise FileNotFoundError(f"Clean parallel corpus not found at {CLEAN_DATA_PATH}. Please run Phase 2 & 3 first to create it.")

# Splitting the larger parallel_df
# First split: 80% Train, 20% Temporary (Val + Test)
train_df, temp_df = train_test_split(parallel_df, test_size=0.2, random_state=42)

# Second split: Split the 20% into half (10% Val, 10% Test)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print row counts
print(f"Total cleaned rows: {len(parallel_df)}")
print(f"--- Splits ---")
print(f"Train set:      {len(train_df)} rows")
print(f"Validation set: {len(val_df)} rows")
print(f"Test set:       {len(test_df)} rows")

# Save to CSV for the next algorithms within the PROJECT_PATH
train_df.to_csv(os.path.join(PROJECT_PATH, 'toy_train.csv'), index=False)
val_df.to_csv(os.path.join(PROJECT_PATH, 'toy_val.csv'), index=False)
test_df.to_csv(os.path.join(PROJECT_PATH, 'toy_test.csv'), index=False)

print("\nSuccess: Scaled datasets split and saved to PROJECT_PATH.")

Total cleaned rows: 8045
--- Splits ---
Train set:      6436 rows
Validation set: 804 rows
Test set:       805 rows

Success: Scaled datasets split and saved to PROJECT_PATH.


This cell defines a rule-based phonetic transliteration algorithm using a predefined mapping of Moroccan Arabizi characters and numbers to their Arabic equivalents. It then tests this algorithm on a few sample phrases.

### Phase 5: Algorithm Implementation
#### Algorithm 1: Rule-Based (Phonetic) Baseline
We create a simple phonetic mapping for common Arabizi characters and numbers to Arabic script.

In [ ]:
import pandas as pd

# 1. Define the Moroccan Arabizi Mapping Dictionary
moroccan_rule_map = {
    '3': 'ع',
    '7': 'ح',
    '9': 'ق',
    '2': 'ء',
    '5': 'خ',
    '8': 'غ',
    '4': 'ش',
    'sh': 'ش',
    'ch': 'ش',
    'kh': 'خ',
    'gh': 'غ',
    'ou': 'و',
    'b': 'ب', 'd': 'د', 'f': 'ف', 'g': 'ق', 'h': 'ه',
    'j': 'ج', 'k': 'ك', 'l': 'ل', 'm': 'م', 'n': 'ن',
    'p': 'ب', 'r': 'ر', 's': 'س', 't': 'ت', 'w': 'و',
    'y': 'ي', 'z': 'ز',
    'a': 'ا',
    'e': '',   # Often silent/unwritten in Arabic script
    'i': 'ي',
    'u': 'و',
    'o': 'و',
}

# 2. The Transliteration Function
def rule_based_transliterate(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    # Sort keys by length (longest first) to handle 'sh' before 's'
    sorted_keys = sorted(moroccan_rule_map.keys(), key=len, reverse=True)

    for key in sorted_keys:
        text = text.replace(key, moroccan_rule_map[key])

    return text

# 3. Test it on a few common phrases
test_phrases = ["finwe", "daba", "ghadi", "bech7al", "mshat", "aji"]
print("--- Algorithm 1 Test ---")
for p in test_phrases:
    print(f"{p} -> {rule_based_transliterate(p)}")

This cell trains a Maximum Likelihood Estimation (MLE) model by calculating word-level probabilities from the training data. It also implements a transliteration function that uses the MLE map with a fallback to the rule-based algorithm for out-of-vocabulary words.

#### Algorithm 2: Statistical Maximum Likelihood Estimation (MLE)
We build a word-level mapping by counting frequency distributions in the training set. If a word is unseen during training, the system falls back to the Rule-Based phonetic map.

In [ ]:
from collections import Counter

def train_mle_probabilities(df):
    # Count occurrences of (Arabizi, Arabic) pairs and total Arabizi counts
    pair_counts = {}
    arabizi_total_counts = Counter()

    for _, row in df.iterrows():
        az_words = str(row['text_az']).lower().split()
        ar_words = str(row['text_ar']).split()

        if len(az_words) == len(ar_words):
            for az_w, ar_w in zip(az_words, ar_words):
                if az_w not in pair_counts:
                    pair_counts[az_w] = Counter()
                pair_counts[az_w][ar_w] += 1
                arabizi_total_counts[az_w] += 1

    # Calculate P(Arabic | Arabizi) and build 1:1 map with the highest probability
    mle_map = {}
    probabilities = {}

    for az_w, counts in pair_counts.items():
        # Find the most frequent Arabic word for this Arabizi word
        most_common_ar, count = counts.most_common(1)[0]
        mle_map[az_w] = most_common_ar
        # Explicitly calculate the probability
        probabilities[az_w] = count / arabizi_total_counts[az_w]

    return mle_map, probabilities

# Train and retrieve probabilities
mle_mapping_table, word_probs = train_mle_probabilities(train_df)

def mle_transliterate_with_fallback(text, mapping_table, fallback_func):
    words = str(text).lower().split()
    translated = []
    for w in words:
        # 1:1 Mapping with Rule-based fallback for OOV
        if w in mapping_table:
            translated.append(mapping_table[w])
        else:
            translated.append(fallback_func(w))
    return " ".join(translated)

# Verification
print(f"Trained on {len(mle_mapping_table)} unique Arabizi tokens.")
sample = "ghadi imchi 3and khouh"
prob = word_probs.get(sample, 0)
print(f"P(Arabic | '{sample}') = {prob:.4f}")
print(f"Transliteration: '{sample}' -> {mle_transliterate_with_fallback(sample, mle_mapping_table, rule_based_transliterate)}")

This cell performs an audit to ensure there is no data leakage in the MLE mapping table, verifying that all unique Arabizi tokens in the MLE model were strictly derived from the training dataset.

In [ ]:
# Data Leakage Audit: Ensure MLE only saw training data
train_tokens = set(' '.join(train_df['text_az'].astype(str)).lower().split())
mle_tokens = set(mle_mapping_table.keys())

leakage = mle_tokens - train_tokens
if not leakage:
    print("✅ Leakage Audit Passed: MLE dictionary is built strictly from the training set.")
else:
    print(f"⚠️ Warning: Found {len(leakage)} tokens in MLE table not in train_df.")

✅ Leakage Audit Passed: MLE dictionary is built strictly from the training set.


This cell compares the transliteration results of the Rule-Based and MLE models on a sample Arabizi sentence, providing a qualitative assessment and token-level analysis of how each word was processed.

In [ ]:
sample_sentence = "mabrouk 3lik maderti, imta l7afla"

# Get results from both models
rule_res = rule_based_transliterate(sample_sentence)
mle_res = mle_transliterate_with_fallback(sample_sentence, mle_mapping_table, rule_based_transliterate)

print(f"Source (Arabizi): {sample_sentence}")
print("-" * 30)
print(f"Algorithm 1 (Rule-Based): {rule_res}")
print(f"Algorithm 2 (MLE + Fallback): {mle_res}")

# Analysis of specific tokens
print("\n--- Token Analysis ---")
for word in sample_sentence.split():
    if word in mle_mapping_table:
        print(f"'{word}' found in MLE training data (P={word_probs.get(word, 0):.2f})")
    else:
        print(f"'{word}' not found in training, used phonetic fallback")

This cell serves as a placeholder for the removed normalization phase. The `camel-tools` installation has been commented out as per the user's request.

### Algorithm 3: Neural Seq2Seq (AraT5)
We implement a character-level Transformer model using **AraT5**. Unlike traditional word-level models, AraT5, similar to ByT5, operates on byte-level tokens, making it robust to the non-standard spelling and character variations found in Moroccan Arabizi. This section handles the model initialization, byte-level preprocessing, and the fine-tuning configuration.

### Phase 5.0.5: CAMeL Arabic Normalization

To improve the consistency and quality of the Arabic target text, we'll use CAMeL Tools for normalization. This step will standardize variations in the Arabic script before tokenization, which can be crucial for neural machine translation models like AraT5.

In [ ]:
# The normalization phase has been removed as requested. Installation of camel-tools is now commented out.
# !pip install camel-tools

This cell defines the `camel_normalizer` function as a passthrough, effectively disabling the Arabic text normalization step as requested by the user, ensuring that Arabic text is processed as-is.

In [ ]:
# The normalization phase has been removed as requested.
# The camel_normalizer function now acts as a passthrough.
def camel_normalizer(text):
    return str(text)

print("CAMeL Arabic Normalizer phase removed. Function now returns text as is.")

This cell initializes the AraT5 tokenizer and defines the `preprocess_function`. This function prepares text data for the AraT5 model by tokenizing inputs (Arabizi) and targets (Arabic), applying truncation and padding, and masking padding tokens in the labels for correct loss calculation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Switching to AraT5 - specialized for Arabic scripts
# Correcting identifier to a valid UBC-NLP model
model_checkpoint = "UBC-NLP/arat5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [str(text) for text in examples["text_az"]]
    targets = [str(text) for text in examples["text_ar"]]

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True, padding="max_length")

    # Masking padding tokens to -100 to ignore them in the loss function
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print(f"Environment switched to AraT5. Preprocessing configured for {model_checkpoint}.")

This cell initializes the AraT5 tokenizer and defines the `preprocess_function`. This function prepares text data for the AraT5 model by tokenizing inputs (Arabizi) and targets (Arabic), applying truncation and padding, and masking padding tokens in the labels for correct loss calculation, specifically without CAMeL Arabic normalization.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Switching to AraT5 - specialized for Arabic scripts
# Correcting identifier to a valid UBC-NLP model
model_checkpoint = "UBC-NLP/arat5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [str(text) for text in examples["text_az"]]
    # Reverting target text processing to remove CAMeL Arabic normalization
    targets = [str(text) for text in examples["text_ar"]]

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True, padding="max_length")

    # Masking padding tokens to -100 to ignore them in the loss function
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print(f"Environment switched to AraT5. Preprocessing configured for {model_checkpoint} (without CAMeL Arabic normalization).")

This cell is a placeholder that was previously used for numpy downgrade. This action is now handled in the comprehensive dependency installation cell `2da20011` and thus this cell is empty.

In [ ]:
# This cell is removed as numpy installation is now handled in cell 2da20011

This cell converts the pandas DataFrames for training and validation into Hugging Face `Dataset` objects. It then applies the `preprocess_function` to tokenize and prepare the datasets for the AraT5 model. It also includes logic to reload and re-split data if not found in RAM.

### Phase 5.1: Dataset Preparation & Tokenization
We convert our CSV splits into Hugging Face datasets and apply the ByT5 byte-level tokenizer.

In [ ]:
from datasets import Dataset, DatasetDict
import gc
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from google.colab import drive

# Ensure Drive is mounted and Path is defined
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/darija_project'
CLEAN_DATA_PATH = os.path.join(PROJECT_PATH, 'clean_parallel_corpus.csv')

# Recovery: Reload and re-split data if missing from RAM
if 'train_df' not in globals():
    print("Dataframes not found in RAM. Reloading from Google Drive...")
    if os.path.exists(CLEAN_DATA_PATH):
        parallel_df = pd.read_csv(CLEAN_DATA_PATH)
        # Re-apply the 80/10/10 split
        train_df, temp_df = train_test_split(parallel_df, test_size=0.2, random_state=42)
        val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    else:
        raise FileNotFoundError(f"Save-point missing at {CLEAN_DATA_PATH}. Please run Phase 2 & 3 first.")

# Ensure tokenizer and preprocess_function are defined
if 'tokenizer' not in globals():
    from transformers import AutoTokenizer
    # Ensuring consistent model_checkpoint for tokenizer
    model_checkpoint = "UBC-NLP/AraT5-base" # Fixed: Consistent model_checkpoint
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Re-apply the map function
train_ds = Dataset.from_pandas(train_df[['text_az', 'text_ar']])
val_ds = Dataset.from_pandas(val_df[['text_az', 'text_ar']])
test_ds = Dataset.from_pandas(test_df[['text_az', 'text_ar']])

tokenized_datasets = DatasetDict({
    'train': train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names),
    'validation': val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
})

# Step 5: Memory Management
del train_df, val_df, test_df, train_ds, val_ds, test_ds
gc.collect()

print(f"Datasets re-tokenized using {model_checkpoint}. Training samples: {len(tokenized_datasets['train'])}")

This cell loads the pre-trained AraT5 model and its tokenizer. It then configures the `Seq2SeqTrainingArguments` with parameters such as learning rate, batch size, evaluation strategy, and saving strategy. Finally, it initializes the `Seq2SeqTrainer` and includes logic to resume training from the latest checkpoint if available.

### Phase 5.2: Training Configuration
Loading the model and setting up the Seq2Seq trainer with mixed precision (`fp16`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import os
import glob
import torch

# The huggingface_hub downgrade is now handled in cell 2da20011
# !pip uninstall huggingface_hub -y
# !pip install huggingface_hub==0.17.3 # Downgrade to a version known to be compatible with older transformers calling conventions

# Fixed: Changed to a valid Hugging Face identifier (AraT5-base)
model_checkpoint = "UBC-NLP/AraT5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# DATA AUDIT
# Check if tokenized_datasets is defined before trying to use it
if 'tokenized_datasets' not in globals():
    raise NameError("The variable 'tokenized_datasets' is not defined. Please ensure cell '2099253a' (Dataset Preparation & Tokenization) has been executed successfully.")

print(f"Training samples: {len(tokenized_datasets['train'])}")
print(f"Validation samples: {len(tokenized_datasets['validation'])}")

training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(PROJECT_PATH, "arat5-arabizi-mor"),
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(), # Safer: Only use fp16 if CUDA is available
    report_to="none",
    logging_steps=10,
    remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

# Logic to resume from the latest checkpoint
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    checkpoints = list(glob.glob(os.path.join(training_args.output_dir, "checkpoint-*")))
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"Resuming training from checkpoint: {last_checkpoint}")


print("Trainer configured for AraT5. Starting training...")
trainer.train(resume_from_checkpoint=last_checkpoint)


This cell plots the training and validation loss curves from the `trainer.state.log_history`. This visualization helps to monitor the model's learning progress and identify potential overfitting or underfitting.

#### Visualization: Training Learning Curves
Plotting the 'Heartbeat' of the AraT5 model.

In [ ]:
# Extract logs from trainer
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

if not train_loss:
    print("Note: No training logs found yet. This usually happens if the training duration is shorter than 'logging_steps' (currently 100).")
else:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, train_loss, label='Training Loss')
    if val_loss:
        # Map eval loss to the correct step (usually at the end of epochs)
        eval_steps = [x['step'] for x in history if 'eval_loss' in x]
        plt.plot(eval_steps, val_loss, 'o-', label='Validation Loss')

    plt.title('ByT5 Training & Validation Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

This cell re-initializes the AraT5 model and trainer. This is a recovery step designed to discard any potentially corrupted or incomplete training progress and restart the training process with a fresh model on valid data.

### Phase 6: Quantitative & Qualitative Evaluation
We evaluate all three models using the BLEU metric and generate comparative visualizations to analyze n-gram precision and model convergence.

### Phase 6: Recovery - Fresh Model Training
Since the previous training run was performed on corrupted/empty inputs, we must re-initialize the `google/byt5-small` weights and run the trainer again with the correct data.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer

# 1. Clear GPU Memory
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

# 2. Re-initialize base model
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 3. Re-initialize Trainer with the same arguments
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

print("Model reset. Starting fresh training run on valid data...")
trainer.train()

This cell performs inference with the freshly trained AraT5 model on the test set. It calculates BLEU scores for all three algorithms (Rule-Based, MLE, and AraT5 Neural) to quantitatively evaluate their performance.

### Phase 7: Re-evaluation
Now we re-run the inference on the test set with the freshly trained weights.

### Phase 7: Quantitative Benchmarking (Final Evaluation)
In this phase, we execute a comparative analysis of the three transliteration approaches—**Rule-Based**, **Statistical MLE**, and **Neural ByT5**—using the **BLEU (Bilingual Evaluation Understudy)** metric on the 10% held-out test set. This provides the objective performance data required for the paper's results section.

In [ ]:
import pandas as pd
import torch
from sacrebleu.metrics import BLEU
import os

# Ensure PROJECT_PATH is defined if not already
if 'PROJECT_PATH' not in globals():
    PROJECT_PATH = '/content/drive/MyDrive/darija_project'

# 1. Recovery: Reload test data and define evaluation metrics
bleu = BLEU()
def evaluate_algorithm(preds, targets):
    score = bleu.corpus_score(preds, [targets])
    return score.score

# Load the test set we saved in Phase 4
test_df_recovery = pd.read_csv(os.path.join(PROJECT_PATH, 'toy_test.csv'))
test_az = test_df_recovery['text_az'].tolist()
test_ar_gold = test_df_recovery['text_ar'].tolist()

# 2. Setup model for inference
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

preds_arat5 = []
print("Running inference with freshly trained AraT5 model...")

for text in test_az:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs["attention_mask"],
            max_new_tokens=64,
            num_beams=4
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds_arat5.append(decoded)

# 3. Calculate Scores (Re-calculating baselines for the report)
# Assuming baseline functions and maps are still in memory
preds_rule = [rule_based_transliterate(text) for text in test_az]
score_rule = evaluate_algorithm(preds_rule, test_ar_gold)

preds_mle = [mle_transliterate_with_fallback(text, mle_mapping_table, rule_based_transliterate) for text in test_az]
score_mle = evaluate_algorithm(preds_mle, test_ar_gold)

score_arat5 = evaluate_algorithm(preds_arat5, test_ar_gold)

print(f"\n--- REFRESHED BLEU Score Results ---")
print(f"Algorithm 1 (Rule-Based): {score_rule:.2f}")
print(f"Algorithm 2 (MLE Baseline): {score_mle:.2f}")
print(f"Algorithm 3 (AraT5 Neural):  {score_arat5:.2f}")

This cell plots the training and validation loss curves from the `trainer.state.log_history`, specifically for Figure 5. This visualization helps confirm the model's convergence and learning behavior.

#### Figure 5: Training Convergence (Loss Curve)
We plot the training and validation loss to confirm the model is learning meaningful patterns from the data.

In [ ]:
import matplotlib.pyplot as plt

# Extract logs safely from the trainer state
history = trainer.state.log_history

# Filter logs for training loss
train_logs = [x for x in history if 'loss' in x]
train_loss = [x['loss'] for x in train_logs]
steps = [x['step'] for x in train_logs]

# Filter logs for validation loss
eval_logs = [x for x in history if 'eval_loss' in x]
val_loss = [x['eval_loss'] for x in eval_logs]
eval_steps = [x['step'] for x in eval_logs]

plt.figure(figsize=(10, 6))

if train_loss:
    plt.plot(steps, train_loss, label='Training Loss', alpha=0.8, marker='.')
else:
    print("Warning: No training loss data found in history.")

if val_loss:
    plt.plot(eval_steps, val_loss, 'o-', label='Validation Loss')

plt.title('Figure 5: AraT5 Training Convergence')
plt.xlabel('Training Steps')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

This cell generates a bar plot to visually compare the final BLEU scores of the Rule-Based, Statistical MLE, and AraT5 Neural transliteration models, corresponding to Figure 6 of the paper.

#### Figure 6: Final BLEU Score Benchmarking
A final comparison of the three implemented algorithms for the paper.

In [ ]:
import seaborn as sns

models = ['Rule-Based (Baseline)', 'Statistical MLE', 'AraT5 Neural']
scores = [score_rule, score_mle, score_arat5]

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=models, y=scores, palette='viridis', hue=models, legend=False)

plt.title('Figure 6: Final BLEU Score Comparison')
plt.ylabel('BLEU Score')
plt.ylim(0, max(scores) + 10)

# Annotate bars with scores
for i, v in enumerate(scores):
    ax.text(i, v + 0.5, f"{v:.2f}", ha='center', fontweight='bold')

plt.show()

This cell performs inference with the trained AraT5 model on the test set, specifically focusing on generating translations with length constraints to prevent brevity penalties. It also calculates and prints the BLEU scores for all three algorithms.

import torch
from sacrebleu.metrics import BLEU

bleu = BLEU()

def evaluate_algorithm(preds, targets):
    score = bleu.corpus_score(preds, [targets])
    return score.score

test_az = test_df['text_az'].tolist()
test_ar_gold = test_df['text_ar'].tolist()

# 1. Prepare Baselines
preds_rule = [rule_based_transliterate(text) for text in test_az]
score_rule = evaluate_algorithm(preds_rule, test_ar_gold)

preds_mle = [mle_transliterate_with_fallback(text, mle_mapping_table, rule_based_transliterate) for text in test_az]
score_mle = evaluate_algorithm(preds_mle, test_ar_gold)

# 2. Updated ByT5 Inference with Length Constraints
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

preds_byt5 = []
print("Running ByT5 inference (Fixed for Brevity Penalty)...")

for text in test_az:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        # We set min_length to prevent the model from immediately outputting EOS
        outputs = model.generate(
            inputs.input_ids,
            max_length=128,
            min_length=10,
            num_beams=5,
            repetition_penalty=2.5,
            early_stopping=True
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds_byt5.append(decoded)

score_byt5 = evaluate_algorithm(preds_byt5, test_ar_gold)

print(f"\n--- BLEU Score Results ---")
print(f"Algorithm 1 (Rule-Based): {score_rule:.2f}")
print(f"Algorithm 2 (MLE Baseline): {score_mle:.2f}")
print(f"Algorithm 3 (ByT5 Neural):  {score_byt5:.2f}")

This cell loads the test data, performs inference with the Rule-Based, MLE, and AraT5 Neural models, and calculates the final BLEU scores for each. This serves as a comprehensive evaluation of all algorithms.

In [ ]:
import torch
import pandas as pd
from sacrebleu.metrics import BLEU
import os

# Ensure PROJECT_PATH is defined if not already
if 'PROJECT_PATH' not in globals():
    PROJECT_PATH = '/content/drive/MyDrive/darija_project'

# Initialize BLEU metric
bleu = BLEU()

def evaluate_algorithm(preds, targets):
    score = bleu.corpus_score(preds, [targets])
    return score.score

# 1. Prepare Test Data (Reloading from disk as test_df was cleared from RAM)
test_df_eval = pd.read_csv(os.path.join(PROJECT_PATH, 'toy_test.csv'))
test_az = test_df_eval['text_az'].tolist()
test_ar_gold = test_df_eval['text_ar'].tolist()

# 2. Baseline 1: Rule-Based
preds_rule = [rule_based_transliterate(text) for text in test_az]
score_rule = evaluate_algorithm(preds_rule, test_ar_gold)

# 3. Baseline 2: MLE
preds_mle = [mle_transliterate_with_fallback(text, mle_mapping_table, rule_based_transliterate) for text in test_az]
score_mle = evaluate_algorithm(preds_mle, test_ar_gold)

# 4. Algorithm 3: AraT5 Inference with expanded byte-limit
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

preds_arat5 = []
print("Running AraT5 inference on test set...")

for text in test_az:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs["attention_mask"],
            max_new_tokens=64,
            num_beams=4
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds_arat5.append(decoded)

score_arat5 = evaluate_algorithm(preds_arat5, test_ar_gold)

# Final Comparison
print("\n--- FINAL BLEU Score Results ---")
print(f"Algorithm 1 (Rule-Based): {score_rule:.2f}")
print(f"Algorithm 2 (MLE Baseline): {score_mle:.2f}")
print(f"Algorithm 3 (AraT5 Neural):  {score_arat5:.2f}")

This cell generates two visualizations: an overall BLEU score comparison using a bar plot and an N-gram precision decay plot for the MLE Baseline and AraT5 Neural models. This provides a detailed analysis of their performance.

#### Visualization: Final Benchmarking
Comparing Algorithm 1, 2, and 3 with N-gram precision decay.

In [ ]:
# 1. Overall Performance Comparison
models = ['Rule-Based', 'MLE Baseline', 'AraT5 Neural']
scores = [score_rule, score_mle, score_arat5]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=scores, palette='magma')
plt.title('Overall BLEU Score Comparison')
plt.ylabel('BLEU Score')
plt.ylim(0, max(scores) + 10)
for i, v in enumerate(scores):
    plt.text(i, v + 0.5, f"{v:.2f}", ha='center', fontweight='bold')
plt.show()

# 2. N-gram Precision Decay
def get_precisions(preds, targets):
    # Extract 1-4 gram precisions from sacrebleu
    result = bleu.corpus_score(preds, [targets])
    return result.precisions

mle_prec = get_precisions(preds_mle, test_ar_gold)
arat5_prec = get_precisions(preds_arat5, test_ar_gold)
ngrams = ['1-gram', '2-gram', '3-gram', '4-gram']

plt.figure(figsize=(10, 6))
plt.plot(ngrams, mle_prec, 's--', label='MLE Baseline', markersize=8)
plt.plot(ngrams, arat5_prec, 'o-', label='AraT5 Neural', markersize=8)
plt.title('N-gram Precision Decay (Context Awareness Proof)')
plt.xlabel('N-gram Order')
plt.ylabel('Precision (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

This cell displays a qualitative failure analysis by comparing a few sample Arabizi inputs with their gold Arabic translations, and the predictions from both the MLE and AraT5 models. This helps in understanding model errors.

In [ ]:
# Qualitative failure analysis: Compare updated examples
comparison_results = pd.DataFrame({
    'Arabizi': test_az[:5],
    'Gold (Arabic)': test_ar_gold[:5],
    'MLE Pred': preds_mle[:5],
    'AraT5 Pred': preds_arat5[:5]
})
display(comparison_results)

This cell inspects a single sample prediction from the AraT5 model, displaying the source Arabizi, gold Arabic, raw output token IDs, and the decoded output. This helps diagnose issues like null characters or incorrect decoding.

In [ ]:
# Inspecting a single sample to diagnose the 0.0 BLEU / null-character issue
sample_idx = 0
sample_text = test_az[sample_idx]
gold_text = test_ar_gold[sample_idx]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=128).to(device)

with torch.no_grad():
    outputs = model.generate(
        inputs.input_ids,
        max_length=128,
        num_beams=5,
        early_stopping=True
    )

print(f"Source Arabizi: {sample_text}")
print(f"Gold Arabic:   {gold_text}")
print(f"Raw Output IDs: {outputs[0].tolist()}")
print(f"Decoded Output: {tokenizer.decode(outputs[0], skip_special_tokens=False)}")

This cell inspects a sample from the tokenized training dataset, showing the tokenized input IDs (Arabizi) and label IDs (Arabic), along with their decoded versions. It also displays the original raw Arabizi and Arabic texts for comparison.

### Inspecting Tokenized Data for Learning Issues
Let's examine a sample from the `tokenized_datasets` to understand why the model might not be learning (0.0 loss, NaN gradients). We'll look at the `input_ids` and `labels` for a single example.

In [ ]:
import pandas as pd
from datasets import Dataset
import os

# Ensure PROJECT_PATH is defined if not already
if 'PROJECT_PATH' not in globals():
    PROJECT_PATH = '/content/drive/MyDrive/darija_project'

# Get a sample from the tokenized training dataset
sample_tokenized_data = tokenized_datasets['train'][0]

print("--- Sample Tokenized Input (text_az) ---")
print(f"Input IDs: {sample_tokenized_data['input_ids'][:20]}...") # Print first 20 for brevity
print(f"Decoded Input: {tokenizer.decode(sample_tokenized_data['input_ids'], skip_special_tokens=True)}")

print("\n--- Sample Tokenized Labels (text_ar) ---")
print(f"Label IDs: {sample_tokenized_data['labels'][:20]}...") # Print first 20 for brevity

# To decode labels, we need to replace -100 with tokenizer.pad_token_id
decodable_labels = [l if l != -100 else tokenizer.pad_token_id for l in sample_tokenized_data['labels']]
print(f"Decoded Labels (after unmasking): {tokenizer.decode(decodable_labels, skip_special_tokens=True)}")

print("\n--- Raw Example from original dataset ---")
# Re-load train_df and create train_ds locally for inspection
train_df_local = pd.read_csv(os.path.join(PROJECT_PATH, 'toy_train.csv'))
train_ds_local = Dataset.from_pandas(train_df_local[['text_az', 'text_ar']])
raw_sample_data = train_ds_local[0]
print(f"Original Arabizi: {raw_sample_data['text_az']}")
print(f"Original Arabic: {raw_sample_data['text_ar']}")

This cell audits the tokenizer's vocabulary to ensure it adequately covers the Arabic characters present in the target (`text_ar`) data from the training set. It identifies any characters that might be missing or mapped to unknown tokens.

### Inspecting Tokenizer Vocabulary for Arabic Characters
Given that the labels are correctly masked, let's verify if the tokenizer's vocabulary adequately covers the Arabic characters present in our target (`text_ar`) data. A mismatch here could lead to the model failing to learn.

In [ ]:
from collections import Counter
import re
import pandas as pd
from datasets import Dataset
import os

# Assuming PROJECT_PATH is defined elsewhere, or define it here if not
# For robustness, we'll ensure PROJECT_PATH is set
if 'PROJECT_PATH' not in globals():
    PROJECT_PATH = '/content/drive/MyDrive/darija_project'

# Re-load train_df and create train_ds_local locally for inspection
train_df_local = pd.read_csv(os.path.join(PROJECT_PATH, 'toy_train.csv'))
train_ds_local = Dataset.from_pandas(train_df_local[['text_az', 'text_ar']])

# Define a regex pattern for Arabic diacritics (harakat)
# This pattern covers most common diacritics in the Arabic Unicode block
HARAKAT_PATTERN = re.compile(r'[\u064B-\u0652\u061A-\u061F\u063B-\u063E\u0653-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7-\u06ED\u06F8-\u06F9\u08E3-\u08EE\u08F0-\u08F3\u08F5-\u08F7]')

# Get all unique relevant characters from the original Arabic training data
arabic_text_characters = Counter()
for text in train_ds_local['text_ar']:
    # Remove diacritics from the entire text for consistent character identification
    text_without_diacritics = HARAKAT_PATTERN.sub('', text)
    for char in text_without_diacritics:
        # Filter out spaces, control characters, and digits, and common punctuation
        if char.strip() and not char.isnumeric() and char not in [' ', '\n', '\t', '\r', '.', ',', '!', '?', ':', ';', '،', '؛', '؟']:
            arabic_text_characters[char] += 1

print("Most common Arabic characters in training data:")
if arabic_text_characters:
    for char, count in arabic_text_characters.most_common(20):
        print(f"'{char}': {count}")
else:
    print("No relevant characters found in Arabic training data after filtering.")

# Check if these characters are in the tokenizer's vocabulary
missing_chars = []
if arabic_text_characters:
    for char in arabic_text_characters:
        # Tokenize each char to see if it's treated as an unknown token
        # For byte-level tokenizers like AraT5, a single character might map to a single token
        # (or potentially multiple if it's a multi-byte character outside ASCII range but represented differently)
        # The key issue here is if it maps to the UNK token.
        tokenized_output = tokenizer.tokenize(char)
        if tokenized_output and tokenized_output[0] == tokenizer.unk_token:
            missing_chars.append(char)

    print(f"\nTotal unique relevant characters in training data: {len(arabic_text_characters)}")
    if missing_chars:
        print(f"\nWARNING: The following {len(missing_chars)} Arabic characters appear to be missing from the tokenizer's vocabulary or are mapped to UNK tokens:")
        print(missing_chars)
    else:
        print("\nAll identified relevant characters appear to be in the tokenizer's vocabulary (not mapped to UNK).")
else:
    print("\nCannot check tokenizer vocabulary as no relevant characters were identified in the training data.")

This cell verifies the AraT5 tokenizer's ability to handle Arabic text by encoding and decoding a sample Arabic string. It also attempts a manual byte-level decoding to confirm the underlying representation.

In [ ]:
# Verification: Can the tokenizer actually handle Arabic bytes?
test_str = "السلام عليكم"
encoded = tokenizer(test_str, return_tensors="pt")
decoded = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=False)

print(f"Target String: {test_str}")
print(f"Token IDs: {encoded['input_ids'][0].tolist()}")
print(f"Decoded String: {decoded}")

# Test raw byte decoding if it looks garbled
raw_ids = encoded['input_ids'][0].tolist()
try:
    # ByT5 uses byte IDs directly (offset by 3 for special tokens usually)
    # Let's see if the IDs correspond to valid UTF-8
    byte_list = bytes([i - 3 for i in raw_ids if i > 3])
    print(f"Manual Byte Decode (UTF-8): {byte_list.decode('utf-8')}")
except Exception as e:
    print(f"Manual byte decode failed: {e}")

This cell generates a summary of the final paper results, including a comparison table of the three transliteration algorithms (Rule-Based, MLE, and AraT5 Neural) and a detailed N-gram precision decay plot for context-awareness analysis.

### Final Paper Results: Performance Summary
This section generates the quantitative evidence for the paper, including the final comparison table and the N-gram decay analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Comparison Table
results_data = {
    'Algorithm': ['Rule-Based (Baseline)', 'Statistical MLE', 'AraT5 Neural'],
    'BLEU Score': [score_rule, score_mle, score_arat5],
    'Methodology': ['Deterministic Phonetic Map', '1:1 Word Frequency', 'Character-level Seq2Seq']
}

results_table = pd.DataFrame(results_data)
print("Table 1: Performance Comparison of Transliteration Algorithms")
display(results_table)

# 2. Detailed N-gram Precision Plot
def get_precisions(preds, targets):
    result = bleu.corpus_score(preds, [targets])
    return result.precisions

rule_prec = get_precisions(preds_rule, test_ar_gold)
mle_prec = get_precisions(preds_mle, test_ar_gold)
arat5_prec = get_precisions(preds_arat5, test_ar_gold)
ngrams = ['1-gram', '2-gram', '3-gram', '4-gram']

plt.figure(figsize=(10, 6))
plt.plot(ngrams, rule_prec, 'v:', label='Rule-Based', alpha=0.6)
plt.plot(ngrams, mle_prec, 's--', label='MLE Baseline')
plt.plot(ngrams, arat5_prec, 'o-', label='AraT5 Neural', linewidth=3)

plt.title('N-gram Precision Decay: Context-Awareness Analysis')
plt.xlabel('N-gram Order')
plt.ylabel('Precision (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()